# MimicVision AI v1.0 - Clase 3 | ASK

Tercera y ultima entrega: integra MIMIC y MATCH en una aplicacion multimodal que indexa
un video, construye una linea temporal de eventos, localiza regiones por lenguaje
natural con LocateAnything-3B, y responde consultas en texto libre con timestamp y
evidencia visual. Incluye tambien un modo en vivo (webcam).

Corre igual en local y en Colab, igual que D1 y D2 -- con una diferencia importante: el
grounding con LocateAnything-3B requiere GPU (idealmente Ampere o superior). Si el
notebook corre sin GPU, esas celdas lo informan claramente y el resto del notebook
sigue funcionando (timeline, evaluacion local, motor de consulta con datos ya
guardados).

In [1]:
import os
from pathlib import Path

try:
    import google.colab  # noqa: F401
    EN_COLAB = True
except ImportError:
    EN_COLAB = False

if EN_COLAB:
    if not Path("mimicvision-ai").exists() and not Path("mimic").exists():
        !git clone https://github.com/USUARIO/mimicvision-ai.git
    if Path("mimicvision-ai").exists():
        os.chdir("mimicvision-ai")
    !pip install -q -r requirements.txt

if Path.cwd().name == "notebooks":
    os.chdir("..")

print("Entorno:", "Colab" if EN_COLAB else "local", "| Carpeta:", Path.cwd().name)

Entorno: local | Carpeta: computer_vision


In [2]:
# El grounding con LocateAnything-3B exige GPU (ver docs/superpowers/specs/
# 2026-07-05-ask-design.md seccion 2: se probaron dos rutas alternativas
# -- auto-hospedar en un T4 y consumir el Space publico anonimo -- y
# ninguna de las dos esta garantizada. Se verifica ahora para decidir
# que celdas de grounding se pueden ejecutar.
import torch

HAY_GPU = torch.cuda.is_available()
print("GPU disponible:", HAY_GPU)
if not HAY_GPU:
    print(
        "Sin GPU: las celdas de grounding con LocateAnything-3B se van a "
        "saltar con un mensaje claro. El resto del notebook (timeline, "
        "evaluacion local, motor de consulta) funciona igual."
    )

GPU disponible: False
Sin GPU: las celdas de grounding con LocateAnything-3B se van a saltar con un mensaje claro. El resto del notebook (timeline, evaluacion local, motor de consulta) funciona igual.


## 1. Video de prueba

Un solo video compuesto por 5 clips con licencia libre de Pexels, concatenados con
letterbox. Como los limites de cada segmento se conocen exactamente, se tiene la verdad
de terreno del timeline sin necesidad de anotacion manual (ver
`data/videos/video_prueba_registro.csv`).

In [3]:
import subprocess
import sys

RUTA_VIDEO_PRUEBA = Path("data/videos/video_prueba.mp4")
RUTA_REGISTRO = Path("data/videos/video_prueba_registro.csv")

if not RUTA_VIDEO_PRUEBA.exists():
    subprocess.run([sys.executable, "-m", "ask.data_curation.construir_video_prueba"], check=True)

import pandas as pd

registro = pd.read_csv(RUTA_REGISTRO)
registro

,segmento,clase_esperada,inicio_s,fin_s,licencia,pagina_origen
0,charla,neutral,0.00,47.00,Pexels License (libre para uso personal y come...,https://www.pexels.com/video/a-man-talking-wit...
1,pulgar,pulgar_arriba,47.00,56.16,Pexels License (libre para uso personal y come...,https://www.pexels.com/video/man-giving-thumbs...
2,cruzados,brazos_cruzados,56.16,64.80,Pexels License (libre para uso personal y come...,https://www.pexels.com/video/people-crossed-ar...
3,saludo,saludo,64.80,77.68,Pexels License (libre para uso personal y come...,https://www.pexels.com/video/a-man-greeting-an...
4,senalar,senalamiento,77.68,82.04,Pexels License (libre para uso personal y come...,https://www.pexels.com/video/man-pointing-at-t...


## 2. Timeline de MIMIC (ruta ligera)

Se muestrea el video a una tasa reducida y se aplica el pipeline de MIMIC ya entrenado
(el mismo `.joblib` de la Clase 1) para construir eventos preliminares. Esta parte corre
igual en cualquier maquina -- no necesita GPU.

In [4]:
from ask.ingest import muestrear_frames_de_video
from ask.timeline import construir_timeline
from mimic.classifier import cargar_modelo
from mimic.landmarks import DetectorHolistic

detector = DetectorHolistic()
modelo = cargar_modelo("models/mimic_clasificador.joblib")

frames_muestreados = list(muestrear_frames_de_video(str(RUTA_VIDEO_PRUEBA), fps_muestreo=8.0))
print(f"{len(frames_muestreados)} frames muestreados de {registro['fin_s'].max():.1f}s de video")

# tamano_ventana=15 se eligio probando 5/15/30/50 contra la verdad de
# terreno (seccion 3): la ventana chica (5) fragmenta el timeline en
# mas de 100 micro-eventos: el clasificador de MIMIC parpadea entre
# clases frame a frame. Ventanas mas grandes (30, 50) tampoco arreglan
# el problema de fondo -- solo añaden retraso en las transiciones. 15
# es el mejor punto intermedio encontrado, no una solucion completa.
eventos_preliminares = construir_timeline(frames_muestreados, detector, modelo, tamano_ventana=15)
print(f"{len(eventos_preliminares)} eventos preliminares detectados")
for evento in eventos_preliminares:
    print(f"  {evento['start_time']:.1f}s - {evento['end_time']:.1f}s: {evento['type']}")

684 frames muestreados de 82.0s de video


43 eventos preliminares detectados
  0.0s - 2.4s: pulgar_arriba
  2.4s - 4.8s: neutral
  4.8s - 6.8s: pulgar_arriba
  6.8s - 7.4s: neutral
  7.4s - 7.9s: pulgar_arriba
  7.9s - 8.2s: neutral
  8.2s - 9.0s: pulgar_arriba
  9.0s - 9.4s: neutral
  9.4s - 11.6s: pulgar_arriba
  11.6s - 13.7s: neutral
  13.7s - 16.6s: pulgar_arriba
  16.6s - 17.9s: neutral
  17.9s - 25.9s: pulgar_arriba
  25.9s - 30.1s: neutral
  30.1s - 32.2s: saludo
  32.2s - 33.8s: neutral
  33.8s - 35.2s: zen
  35.2s - 37.0s: neutral
  37.0s - 37.2s: pulgar_arriba
  37.2s - 37.6s: neutral
  37.6s - 37.7s: pulgar_arriba
  37.7s - 38.5s: saludo
  38.5s - 39.1s: neutral
  39.1s - 39.2s: saludo
  39.2s - 40.4s: pulgar_arriba
  40.4s - 42.2s: saludo
  42.2s - 43.8s: neutral
  43.8s - 48.0s: brazos_cruzados
  48.0s - 50.4s: neutral
  50.4s - 52.9s: pulgar_arriba
  52.9s - 54.5s: brazos_cruzados
  54.5s - 54.8s: pensando
  54.8s - 56.4s: pulgar_arriba
  56.4s - 58.8s: brazos_cruzados
  58.8s - 62.2s: pulgar_arriba
  62.2s - 63

## 3. Evaluacion del timeline contra la verdad de terreno

Esta evaluacion es 100% local: se compara el timeline detectado contra los limites
reales de cada segmento del video compuesto. Por cada segmento de la verdad de terreno,
se mide si el evento dominante detectado en ese rango de tiempo coincide con la clase
esperada -- una medida directa de "tasa de acierto de eventos" que pide el docx, sin
necesitar anotacion manual adicional.

In [5]:
from collections import Counter


def clase_dominante_en_rango(eventos, inicio, fin):
    duracion_por_clase = Counter()
    for evento in eventos:
        solapa_inicio = max(evento["start_time"], inicio)
        solapa_fin = min(evento["end_time"], fin)
        if solapa_fin > solapa_inicio:
            duracion_por_clase[evento["type"]] += solapa_fin - solapa_inicio
    if not duracion_por_clase:
        return None
    return duracion_por_clase.most_common(1)[0][0]


aciertos = 0
filas_evaluacion = []
for fila in registro.itertuples():
    detectada = clase_dominante_en_rango(eventos_preliminares, fila.inicio_s, fila.fin_s)
    acierto = detectada == fila.clase_esperada
    aciertos += acierto
    filas_evaluacion.append({
        "segmento": fila.segmento,
        "clase_esperada": fila.clase_esperada,
        "clase_detectada": detectada,
        "acierto": acierto,
    })

tasa_acierto = aciertos / len(registro)
print(f"Tasa de acierto de eventos: {tasa_acierto:.1%} ({aciertos}/{len(registro)})")
pd.DataFrame(filas_evaluacion)

Tasa de acierto de eventos: 40.0% (2/5)


,segmento,clase_esperada,clase_detectada,acierto
0,charla,neutral,pulgar_arriba,False
1,pulgar,pulgar_arriba,pulgar_arriba,True
2,cruzados,brazos_cruzados,brazos_cruzados,True
3,saludo,saludo,pulgar_arriba,False
4,senalar,senalamiento,pulgar_arriba,False


## 4. Búsqueda visual heredada de MATCH

Se reutiliza el índice de MATCH (Recall@5 ya validado en la Clase 2) para consultar,
por ejemplo, el frame representativo de cada evento contra la galería de MIMIC.

In [6]:
import cv2
import numpy as np

from match.embeddings import ExtractorSigLIP2
from match.gallery import cargar_galeria_y_consultas
from match.index import IndiceSimilitud

galeria_match, _ = cargar_galeria_y_consultas()
extractor_match = ExtractorSigLIP2()

metadatos_galeria_match = []
vectores_galeria_match = []
for fila in galeria_match.itertuples():
    imagen = cv2.imread(fila.ruta)
    if imagen is None:
        continue
    vectores_galeria_match.append(extractor_match.extraer(imagen))
    metadatos_galeria_match.append({
        "image_id": fila.sample_id, "etiqueta_pose": fila.clase, "ruta_archivo": fila.ruta,
    })

indice_match = IndiceSimilitud(np.array(vectores_galeria_match), metadatos_galeria_match)

# consulta de ejemplo: el frame representativo del primer evento detectado
resultado_busqueda = indice_match.buscar(
    extractor_match.extraer(eventos_preliminares[0]["frame_representativo"]), k=5
)
[r["etiqueta_pose"] for r in resultado_busqueda]

C:\Users\migue\Downloads\computer_vision\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[transformers] Model config: bos_token_id must be `None` or an integer within the vocabulary (between 0 and 31999), got 49406. This may result in unexpected behavior.


[transformers] Model config: eos_token_id must be `None` or an integer within the vocabulary (between 0 and 31999), got 49407. This may result in unexpected behavior.


Loading weights:   0%|          | 0/408 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 408/408 [00:00<00:00, 11566.35it/s]

['pulgar_arriba', 'neutral', 'saludo', 'saludo', 'pensando']

## 5. Grounding con LocateAnything-3B

Esta seccion solo corre con GPU. El codigo sigue el uso documentado en la model card de
`nvidia/LocateAnything-3B`; no se pudo validar en este entorno (ver seccion 2 del spec de
ASK). Si no hay GPU, se informa claramente y se continua sin romper el notebook.

In [7]:
from ask.grounding import ClienteLocateAnything, ErrorHardwareNoCompatible

PROMPTS_MINIMOS = [
    "person with arms crossed",
    "person giving a thumbs up",
    "person waving hello",
    "person pointing",
    "all people in the scene",
]

cliente_grounding = None
if HAY_GPU:
    try:
        cliente_grounding = ClienteLocateAnything()
        print("LocateAnything-3B cargado correctamente")
    except ErrorHardwareNoCompatible as error:
        print(f"No se pudo cargar LocateAnything-3B: {error}")
else:
    print("Sin GPU: se omite la carga de LocateAnything-3B en esta ejecucion.")

Sin GPU: se omite la carga de LocateAnything-3B en esta ejecucion.


In [8]:
resultados_grounding = []
if cliente_grounding is not None:
    for evento, prompt in zip(eventos_preliminares, PROMPTS_MINIMOS):
        resultado = cliente_grounding.localizar(evento["frame_representativo"], prompt)
        resultados_grounding.append(resultado)
        print(f"{prompt!r} -> {len(resultado['cajas'])} caja(s)")
else:
    print("Grounding no disponible en esta ejecucion (sin GPU compatible).")

Grounding no disponible en esta ejecucion (sin GPU compatible).


## 6. Almacén de eventos y motor de consulta

Los eventos (con o sin grounding, según si hubo GPU disponible) se guardan en
`event_store.json`. El motor de consulta recibe el cliente de grounding por parámetro
-- si no hay GPU, se puede seguir probando la lógica de selección de eventos con un
cliente simulado, tal como en los tests de `ask/query_engine.py`.

In [9]:
from ask.events import construir_event
from ask.store import guardar_eventos

eventos_finales = [construir_event(e, fuente="MIMIC") for e in eventos_preliminares]
guardar_eventos(eventos_finales, "data/event_store.json")
print(f"{len(eventos_finales)} eventos guardados en data/event_store.json")

43 eventos guardados en data/event_store.json


In [10]:
from ask.query_engine import responder_consulta

if cliente_grounding is not None:
    respuesta = responder_consulta("person with arms crossed", eventos_preliminares, cliente_grounding)
    print(respuesta)
else:
    print("Motor de consulta con LocateAnything real: requiere GPU (Colab).")

Motor de consulta con LocateAnything real: requiere GPU (Colab).


## 7. Modo en vivo (webcam)

En vivo, MIMIC corre continuo sobre la webcam y el event store crece con el tiempo. El
usuario puede escribir una consulta en cualquier momento contra los eventos acumulados
hasta ese punto -- misma lógica de `responder_consulta`, solo cambia la fuente de
frames (`ask.ingest.muestrear_frames_en_vivo` en vez de `muestrear_frames_de_video`).
Esta celda no se ejecuta en el notebook (necesita una cámara real); se deja documentada
para correr en local con `python -m app.app`.

In [11]:
# Ejemplo de uso en vivo (no se ejecuta aqui, requiere camara):
#
# from mimic.capture import leer_frames_de_webcam
# from ask.ingest import muestrear_frames_en_vivo
#
# frames_en_vivo = muestrear_frames_en_vivo(leer_frames_de_webcam())
# eventos_en_vivo = construir_timeline(frames_en_vivo, detector, modelo)
print("Ver comentario: el modo en vivo se prueba desde la demo Gradio (python -m app.app).")

Ver comentario: el modo en vivo se prueba desde la demo Gradio (python -m app.app).


## 8. Demo Gradio

Pestaña ASK: elegir video o modo en vivo, escribir una consulta, ver la respuesta con
timestamp y frame de evidencia.

In [12]:
from app.app import construir_demo

construir_demo().launch(share=EN_COLAB, prevent_thread_lock=True)

* Running on local URL:  http://127.0.0.1:7860


* To create a public link, set `share=True` in `launch()`.


## Cierre del proyecto integral

MimicVision AI queda completo: MIMIC clasifica poses y gestos en tiempo real, MATCH
recupera contenido visual por similitud, y ASK integra ambos con grounding por lenguaje
natural y una línea temporal de eventos, en modo batch y en vivo. Las limitaciones reales
encontradas en cada etapa (volumen de datos, sesgo hacia clases grandes, requisitos de
hardware de LocateAnything-3B) están documentadas en los informes de cada clase en vez
de ocultarse.